# Autoscaling Experiment Analysis — Thành

Dữ liệu thực nghiệm từ các lượt chạy HPA CPU, KEDA raw và KEDA optimized. Không tạo dữ liệu giả. Mỗi dòng `metadata.json` tương ứng một lần lặp. **Thiếu metrics được hiển thị là NaN, không tự coi là 0.**

In [ ]:
from pathlib import Path
import json
import math
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data"
FIGURES = ROOT / "notebooks" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
POLICIES = ["baseline-cpu", "custom-unoptimized", "custom-optimized"]
print("Data root:", DATA.resolve())

In [ ]:
def k6_stat(summary, metric, key, fallback=float("nan")):
    block = summary.get("metrics", {}).get(metric, {})
    return block.get("values", {}).get(key, fallback)

records = []
for policy in POLICIES:
    for metadata_path in sorted((DATA / policy).glob("*/metadata.json")):
        folder = metadata_path.parent
        summary_path = folder / "k6-summary.json"
        if not summary_path.exists():
            print("Missing k6 summary:", folder)
            continue
        try:
            meta = json.loads(metadata_path.read_text(encoding="utf-8-sig"))
            summary = json.loads(summary_path.read_text(encoding="utf-8-sig"))
        except (ValueError, OSError) as err:
            print("Invalid run:", folder, err)
            continue
        duration = meta.get("ended_unix", 0) - meta.get("started_unix", 0)
        total = k6_stat(summary, "http_reqs", "count")
        errors = k6_stat(summary, "app_error_rate", "rate")
        if math.isnan(errors):
            errors = k6_stat(summary, "http_req_failed", "rate")
        row = {"policy": meta.get("policy", policy), "scenario": meta.get("scenario", "unknown"),
               "endpoint": meta.get("endpoint", "unknown"), "repeat": int(meta.get("repeat", 1)), "run": folder.name,
               "rate": meta.get("rate", float("nan")), "max_vus": meta.get("max_vus", float("nan")),
               "work_time": meta.get("work_time", float("nan")), "iterations": meta.get("iterations", float("nan")),
               "max_queue_limit": meta.get("max_queue_limit", meta.get("queue_limit", float("nan"))),
               "sample_seconds": meta.get("sample_seconds", 5),
               "load_balanced": meta.get("load_balanced", False),
               "safety_stop": meta.get("safety_stop"),
               "duration_s": duration, "requests": total,
               "throughput_rps": k6_stat(summary, "http_reqs", "rate"),
               "p95_ms": k6_stat(summary, "http_req_duration", "p(95)"),
               "error_rate": errors,
               "dropped_iterations": k6_stat(summary, "dropped_iterations", "count", 0),
               "overload_503": k6_stat(summary, "overload_503_total", "count", 0),
               "cpu_cores_avg": float("nan"), "memory_mib_avg": float("nan"),
               "replicas_peak": float("nan"), "replicas_avg": float("nan"),
               "scale_out_s": float("nan"), "scale_in_s": float("nan"), "replica_changes": float("nan")}
        metrics_file = folder / "prometheus_timeseries.csv"
        if metrics_file.exists():
            ts = pd.read_csv(metrics_file)
            if not ts.empty:
                for col in ["cpu_cores", "memory_bytes", "replicas_available", "replicas_desired"]:
                    if col in ts:
                        ts[col] = pd.to_numeric(ts[col], errors="coerce")
                if "cpu_cores" in ts:
                    row["cpu_cores_avg"] = ts.cpu_cores.mean()
                if "memory_bytes" in ts:
                    row["memory_mib_avg"] = ts.memory_bytes.mean() / (1024**2)
                replica_col = "replicas_available" if "replicas_available" in ts and ts.replicas_available.notna().any() else "replicas_desired"
                if replica_col in ts and ts[replica_col].notna().any():
                    valid = ts[["timestamp", replica_col]].dropna().sort_values("timestamp")
                    row["replicas_peak"] = valid[replica_col].max()
                    row["replicas_avg"] = valid[replica_col].mean()
                    row["replica_changes"] = valid[replica_col].diff().fillna(0).ne(0).sum()
                    origin = meta.get("started_unix", valid.timestamp.iloc[0])
                    baseline = valid.loc[valid.timestamp <= origin, replica_col]
                    initial = baseline.iloc[-1] if not baseline.empty else valid[replica_col].iloc[0]
                    up = valid[(valid.timestamp >= origin) & (valid[replica_col] > initial)]
                    if not up.empty:
                        row["scale_out_s"] = float(up.timestamp.iloc[0] - origin)
                        after_up = valid[valid.timestamp > up.timestamp.iloc[0]]
                        down = after_up[after_up[replica_col] <= initial]
                        if not down.empty:
                            row["scale_in_s"] = float(down.timestamp.iloc[0] - up.timestamp.iloc[0])
        records.append(row)
runs = pd.DataFrame.from_records(records)
if runs.empty:
    print("CHƯA CÓ DỮ LIỆU THỰC NGHIỆM. Chạy scripts/run-experiment.ps1 trước.")
else:
    display(runs.sort_values(["scenario", "endpoint", "policy", "repeat"]))
    runs.to_csv(DATA / "comparison-runs.csv", index=False)

In [ ]:
if not runs.empty:
    aggregate = (runs.groupby(["scenario", "endpoint", "policy"], dropna=False)
                 .agg(repeats=("run", "count"), p95_ms=("p95_ms", "mean"),
                      throughput_rps=("throughput_rps", "mean"),
                      error_rate=("error_rate", "mean"),
                      cpu_cores_avg=("cpu_cores_avg", "mean"),
                      replicas_avg=("replicas_avg", "mean"),
                      replicas_peak=("replicas_peak", "max"),
                      replica_changes=("replica_changes", "mean"),
                      scale_out_s=("scale_out_s", "mean"),
                      dropped_iterations=("dropped_iterations", "sum"))
                 .reset_index())
    display(aggregate)
    aggregate.to_csv(DATA / "comparison-aggregated.csv", index=False)
else:
    aggregate = pd.DataFrame()

In [ ]:
if not aggregate.empty:
    for (scenario, endpoint), subset in aggregate.groupby(["scenario", "endpoint"]):
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.bar(subset.policy, subset.p95_ms)
        ax.set(title=f"p95 latency — {scenario} / {endpoint}", ylabel="p95 latency (ms)", xlabel="Autoscaling policy")
        ax.tick_params(axis="x", rotation=15)
        fig.tight_layout()
        fname = f"p95-{scenario}-{endpoint}"
        fig.savefig(FIGURES / f"{fname}.png", dpi=200)
        fig.savefig(FIGURES / f"{fname}.pdf")
        plt.show()

    for meta_file in sorted(DATA.glob("*/*/metadata.json")):
        timeline = meta_file.parent / "prometheus_timeseries.csv"
        if not timeline.exists():
            continue
        df = pd.read_csv(timeline)
        if df.empty: continue
        col = next((k for k in ("replicas_available", "replicas_desired") if k in df and df[k].notna().any()), None)
        if col is None: continue
        fig, ax = plt.subplots(figsize=(10, 4))
        x = pd.to_datetime(df.timestamp, unit="s", utc=True)
        ax.step(x, pd.to_numeric(df[col], errors="coerce"), where="post")
        ax.set(title=f"Replica timeline — {meta_file.parent.name}", xlabel="Time (UTC)", ylabel="Replicas")
        fig.autofmt_xdate()
        fig.tight_layout()
        fig.savefig(FIGURES / f"replicas-{meta_file.parent.name}.png", dpi=200)
        fig.savefig(FIGURES / f"replicas-{meta_file.parent.name}.pdf")
        plt.close(fig)
    print("Exported figures to:", FIGURES)
else:
    print("No charts until actual runs are available.")

## Giới hạn phương pháp

- `p95_ms` lấy trực tiếp từ thống kê k6, không tính trung bình p95 rồi tuyên bố là p95 toàn bộ yêu cầu.
- `scale_out_s` = thời điểm metric replica đầu tiên lớn hơn replica tại đầu lượt chạy; bao gồm độ trễ lấy mẫu và không xác định nguyên nhân riêng rẽ.
- `scale_in_s` = từ scale-out đầu tiên đến lúc trở lại số replica ban đầu, **không phải cooldown sau khi tải kết thúc**.
- `replica_changes` đếm số thay đổi, **không tự động đồng nghĩa với flapping**. Đếm flapping chính xác cần định nghĩa cửa sổ thời gian và chiều đảo tăng/giảm.
- Nếu thiếu kube-state-metrics hoặc cAdvisor, các trường replicas/CPU/RAM có thể NaN.
- Cần đối chiếu tỷ lệ lỗi, dropped iterations và tải thực đạt được trước khi kết luận chính sách nào tốt hơn.
